# Micro-Projeto de Visão Computacional — MODELO

**Capacitação em Visão Computacional** · IFPE · INPaR · Financiamento FACEPE

Faça uma cópia (**Arquivo → Salvar uma cópia no Drive**) e troque o exemplo pelo **seu** projeto, mantendo as seções.
**Entrega:** este notebook + vídeo de até 2 min, pelo Classroom, até **domingo, 11/10, 23h59**.
O exemplo abaixo (contador de moedas) mostra o padrão esperado: **problema → dados → pipeline → avaliação → ética → conclusão**.

**Equipe:** _(nome(s))_   ·   **Título:** _(nome do projeto)_

## 1. Problema

_Qual problema, para quem e por quê (1–2 frases)?_

**Exemplo:** contar automaticamente as moedas de uma imagem, mesmo com variações de iluminação e qualidade.

## 2. Preparação

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import data          # imagens de exemplo (troque pelas suas)

## 3. Dados

_De onde vêm as imagens ou vídeos? Quantos? Em que condições?_

**Exemplo:** a imagem de moedas em 6 condições (luz, sombra, ruído, desfoque). Gabarito: **24 moedas** em todas.

In [ ]:
moedas = data.coins()                               # imagem em tons de cinza
rng = np.random.default_rng(0)                      # sorteios reprodutíveis
sombra = np.tile(np.linspace(-60, 60, moedas.shape[1]), (moedas.shape[0], 1))   # luz desigual

casos = {                                           # nome -> (imagem, gabarito)
    "original":       (moedas, 24),
    "mais escura":    (np.clip(moedas * 0.6, 0, 255).astype(np.uint8), 24),
    "mais clara":     (np.clip(moedas.astype(int) + 50, 0, 255).astype(np.uint8), 24),
    "sombra lateral": (np.clip(moedas + sombra, 0, 255).astype(np.uint8), 24),
    "ruído":          (np.clip(moedas + rng.normal(0, 20, moedas.shape), 0, 255).astype(np.uint8), 24),
    "desfocada":      (cv2.GaussianBlur(moedas, (9, 9), 0), 24),
}
fig, eixos = plt.subplots(1, len(casos), figsize=(18, 3))
for ax, (nome, (im, _)) in zip(eixos, casos.items()):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255); ax.set_title(nome); ax.axis("off")
plt.show()

### Usar as suas imagens (Colab)

Envie as imagens e preencha o gabarito à mão (ex.: quantos objetos há em cada uma).

In [ ]:
from google.colab import files
enviados = files.upload()                                    # escolha várias imagens
casos = {nome: (cv2.imread(nome, cv2.IMREAD_GRAYSCALE), None) for nome in enviados}
# Depois, preencha o gabarito: casos["foto1.jpg"] = (casos["foto1.jpg"][0], 12)

## 4. Pipeline

_Que técnicas do curso? Que medida sai no final?_ Escreva tudo numa função `processar(imagem)`.

In [ ]:
def processar(img):
    """Recebe uma imagem em cinza e devolve o número de moedas."""
    suave = cv2.GaussianBlur(img, (5, 5), 0)                         # 23/09: tira ruído
    binaria = cv2.adaptiveThreshold(suave, 255,
                                    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,  # 23/09: limiar por região
                                    cv2.THRESH_BINARY,
                                    71,                               # blockSize (ímpar)
                                    -5)                               # C
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    binaria = cv2.morphologyEx(binaria, cv2.MORPH_OPEN, kernel)       # 24/09: limpa ruído
    cnts, _ = cv2.findContours(binaria, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)   # 24/09
    return sum(cv2.contourArea(c) > 400 for c in cnts)                # descarta regiões pequenas

print("original:", processar(casos["original"][0]), "moedas")

## 5. Avaliação

Compare o resultado com o gabarito em **todos** os casos. Inclua casos difíceis.

In [ ]:
def avaliar(funcao, casos):
    linhas = []
    for nome, (img, gabarito) in casos.items():
        previsto = funcao(img)
        linhas.append((nome, gabarito, previsto, abs(previsto - gabarito)))
    print(f"{'caso':16s}{'gabarito':>9s}{'previsto':>9s}{'erro':>6s}")
    for nome, g, p, e in linhas:
        print(f"{nome:16s}{g:9d}{p:9d}{e:6d}")
    acertos = sum(e == 0 for *_, e in linhas)
    erro_medio = np.mean([e for *_, e in linhas])
    print(f"\nacertos exatos: {acertos}/{len(linhas)} | erro médio: {erro_medio:.2f}")
    return linhas

resultado = avaliar(processar, casos)

**Comparação com uma alternativa** (mostra por que escolhemos o limiar adaptativo):

In [ ]:
def processar_otsu(img):
    suave = cv2.GaussianBlur(img, (5, 5), 0)
    _, binaria = cv2.threshold(suave, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)   # um limiar só
    cnts, _ = cv2.findContours(binaria, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return sum(cv2.contourArea(c) > 400 for c in cnts)

_ = avaliar(processar_otsu, casos)

## 6. Ética

_Aparecem pessoas? Houve consentimento? O que é guardado? Para quem o sistema pode errar mais?_

**Exemplo:** não há pessoas nas imagens. Risco: funcionar pior com moedas de outros países ou fundos diferentes — seria preciso testar.

## 7. Conclusão

_O que funcionou, onde falha e o que faria com mais tempo (3–5 frases)._

**Exemplo:** o limiar adaptativo acertou as 24 moedas nas 6 condições; o Otsu errou em todas, chegando a 17 com sombra.
O método ainda não foi testado com moedas encostadas umas nas outras — seria o próximo teste.